# WM Prediction 2026 – Basismodell v1

Ziele dieser Version:

- Nur relevante Länderspiele der letzten 10 Jahre
- Unterstützung aller Kontinentalturniere
- Neutrale WM-Spiele (kein Heimvorteil)
- Zeitgewichtung für aktuelle Spiele
- Turniergewichtung
- Poisson-Modell als Basis für exakte Ergebniswahrscheinlichkeiten


In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import poisson
from datetime import datetime


## Daten laden

In [3]:
RESULTS_FILE = "results.csv"

matches = pd.read_csv(RESULTS_FILE)
matches['date'] = pd.to_datetime(matches['date'])

matches.head()


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


## Nur letzte 10 Jahre verwenden

In [4]:
cutoff = pd.Timestamp.now() - pd.DateOffset(years=10)
matches = matches[matches['date'] >= cutoff].copy()

print(len(matches))


9577


## Turniergewichtung

In [5]:
def tournament_weight(tournament):

    t = str(tournament)

    if 'FIFA World Cup' in t:
        return 20

    if 'UEFA Euro' in t:
        return 15

    if 'Copa América' in t:
        return 15

    if 'African Cup of Nations' in t:
        return 15

    if 'AFC Asian Cup' in t:
        return 15

    if 'OFC Nations Cup' in t:
        return 15

    if 'Nations League' in t:
        return 10

    if 'qualification' in t.lower():
        return 8

    if 'Friendly' in t:
        return 1

    return 3

matches['tournament_weight'] = matches['tournament'].apply(tournament_weight)


## Zeitgewichtung

In [6]:
today = pd.Timestamp.now()

days_old = (today - matches['date']).dt.days

matches['time_weight'] = np.exp(-days_old / 365)

matches['weight'] = (
    matches['tournament_weight']
    * matches['time_weight']
)


## Datensatz für Poisson-Modell erzeugen

In [7]:
home = matches[['home_team','away_team','home_score','weight']].copy()
home.columns = ['team','opponent','goals','weight']

away = matches[['away_team','home_team','away_score','weight']].copy()
away.columns = ['team','opponent','goals','weight']

goal_model_data = pd.concat([home, away], ignore_index=True)

goal_model_data.head()


,team,opponent,goals,weight
0,Belgium,Norway,3.0,0.000045
1,Central African Republic,Angola,3.0,0.000679
2,Comoros,Burkina Faso,0.0,0.000679
3,Czech Republic,South Korea,1.0,0.000045
4,Jamaica,Venezuela,0.0,0.000679


## Poisson-Modell trainieren

In [8]:
model = smf.glm(
    formula='goals ~ team + opponent',
    data=goal_model_data,
    family=sm.families.Poisson(),
    freq_weights=goal_model_data['weight']
).fit()

print(model.summary())


                 Generalized Linear Model Regression Results                  
Dep. Variable:                  goals   No. Observations:                19010
Model:                            GLM   Df Residuals:                 18288.81
Model Family:                 Poisson   Df Model:                          581
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -24853.
Date:                Thu, 04 Jun 2026   Deviance:                       18185.
Time:                        16:46:20   Pearson chi2:                 1.63e+04
No. Iterations:                    18   Pseudo R-squ. (CS):             0.5114
Covariance Type:            nonrobust                                         
                                                   coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------

## Erwartete Tore berechnen

In [9]:
def expected_goals(team, opponent):

    df = pd.DataFrame({
        'team':[team],
        'opponent':[opponent]
    })

    return float(model.predict(df)[0])


## Ergebniswahrscheinlichkeiten

In [10]:
def match_matrix(team1, team2, max_goals=8):

    lam1 = expected_goals(team1, team2)
    lam2 = expected_goals(team2, team1)

    matrix = np.outer(
        poisson.pmf(range(max_goals + 1), lam1),
        poisson.pmf(range(max_goals + 1), lam2)
    )

    return matrix


In [11]:
def top_results(team1, team2, top_n=10):

    matrix = match_matrix(team1, team2)

    results = []

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            results.append((i, j, matrix[i, j]))

    results.sort(key=lambda x: x[2], reverse=True)

    return pd.DataFrame(
        results[:top_n],
        columns=['team1_goals', 'team2_goals', 'probability']
    )


In [14]:
# Beispiel

top_results('Germany', 'Ecuador')


,team1_goals,team2_goals,probability
0,0,0,0.303038
1,1,0,0.187670
2,0,1,0.174126
3,1,1,0.107836
4,2,0,0.058112
5,0,2,0.050027
6,2,1,0.033391
7,1,2,0.030981
8,3,0,0.011996
9,2,2,0.009593


# Nächste Ausbaustufen

1. Elo Ratings
2. FIFA Ranking
3. Form der letzten 5 Spiele
4. Automatischer Download neuer Ergebnisse
5. Wettquoten
6. Opta-Wahrscheinlichkeiten
7. Ensemble-Modell
8. Monte-Carlo-WM-Simulation
